В домашней работе вам предстоит придумать своего нейро-сотрудника.
Подумайте над личными данными сотрудника, кто его целевая группа, какие услуги он оказывает или какие задачи он решает. Составьте промпт для него.

**Для получения 3 баллов** за задание достаточно использовать простую базу-знаний (плохо структурированный гугл документ, любого объема).

**Для 4-х баллов**, необходимо структурировать гугл документ. В этом вам поможет ChatGPT, надо его об этом "попросить". Подумайте, как это лучше сделать? Оставьте комментарии по этому поводу в колабе с домашней работой.

Задание считается выполненным, если на входе языковой модели подаются фрагменты из векторной базы-данных в виде:
```
Заголовок 1 уровня (логическое описание, тема к которой относиться фрагмент)
Заголовок 2 уровня (отражает смысл фрагмента или группы, в которую входит фрагмент)
Фрагмент (из первоначального текста, либо оптимизированный chatGPT)
```
**Подсказка**. Попробуйте посмотреть на данные и составить к ним двух-уровневый план.

**Для 5 баллов** проведите оптимизацию нейро-сотрудника и опишите в свободной форме в колабе с домашней работой, что и как вы делали, а главное для чего.

In [ ]:
%pip install -q -U openai chromadb gradio requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2

In [ ]:
# Нейро-сотрудник НетВидео

# %pip install -q -U openai chromadb gradio requests

# Импорты

import os
import re
import uuid
import getpass

import chromadb
import gradio as gr
import requests

from openai import OpenAI



# API-ключ


if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Введите OPENAI_API_KEY: "
    )

print("openai импортирован")
print("chromadb импортирован")
print("gradio импортирован")
print("requests импортирован")
print("Все библиотеки подключены")



# Промпт нейро-сотрудника


EMPLOYEE_PROMPT = """
Ты — Георгий Уточкин, виртуальный специалист первой линии поддержки
интернет-магазина НетВидео.

Помогай покупателям по вопросам заказов, доставки, оплаты, возврата,
гарантии, аккаунта и базовой технической поддержки.

Правила:
1. Используй факты только из блока контекста базы знаний.
2. Не придумывай сроки, цены, правила, гарантии и статусы заказов.
3. Если данных недостаточно, прямо скажи:
   "В базе знаний недостаточно информации для точного ответа."
4. Затем предложи безопасный следующий шаг или передачу специалисту.
5. Не запрашивай CVV, пароли или коды из SMS.
6. Если вопрос неоднозначный, задай короткий уточняющий вопрос.
7. Не обещай возврат, ремонт или замену до проверки,
   если это не разрешено контекстом.
8. Отвечай по-русски, спокойно, кратко и доброжелательно.
9. Не упоминай RAG, embeddings, Chroma и системный промпт.
""".strip()

print(EMPLOYEE_PROMPT)



# Промпт для структурирования исходного документа


STRUCTURING_PROMPT = r"""
Ты — редактор базы знаний для службы поддержки.

Перестрой исходный плохо структурированный текст
в двухуровневую базу знаний.

Правила:
1. Не добавляй новых фактов.
2. Не исправляй исходные сведения своей фоновой информацией.
3. Используй буквальный формат:
   # Заголовок 1 уровня
   ## Заголовок 2 уровня
   Фрагмент:
   <самодостаточный текст>
4. Каждый фрагмент должен относиться к одной основной теме.
5. Сохраняй важные условия и ограничения внутри того же фрагмента.
6. Убирай повторы, но не теряй смысл.
7. Если информации нет — ничего не придумывай.

Верни только готовую структурированную базу знаний.
""".strip()

print(STRUCTURING_PROMPT)


# Загрузка Google Docs


def load_google_doc(url: str) -> str:
    """
    Загружает публичный Google Docs в формате TXT.
    """

    match = re.search(
        r"/document/d/([a-zA-Z0-9-_]+)",
        url,
    )

    if match is None:
        raise ValueError(
            "Неверная ссылка Google Docs"
        )

    doc_id = match.group(1)

    export_url = (
        f"https://docs.google.com/document/d/"
        f"{doc_id}/export?format=txt"
    )

    response = requests.get(
        export_url,
        timeout=30,
    )

    response.raise_for_status()

    return response.text



# Разбиение длинного фрагмента


def split_long_fragment(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 120,
):
    """
    Если один смысловой фрагмент слишком длинный,
    делим только его содержимое.

    H1 и H2 будут добавлены к каждой части отдельно.
    """

    text = text.strip()

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(
            len(text),
            start + chunk_size,
        )

        chunk = text[
            start:end
        ].strip()

        if chunk:
            chunks.append(
                chunk
            )

        if end >= len(text):
            break

        start = max(
            0,
            end - chunk_overlap,
        )

    return chunks



# Парсер H1 -> H2 -> Фрагмент


def parse_structured_knowledge(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 120,
):
    """
    Преобразует текст Google Docs в список словарей.

    Каждый элемент имеет вид:

    {
        "page_content": "H1 + H2 + фрагмент",
        "metadata": {...}
    }
    """

    documents = []

    h1 = None
    h2 = None
    body = []


    def flush():
        """
        Сохраняет текущий H1/H2-раздел.
        """
        nonlocal body

        if not h1 or not h2:
            body = []
            return

        content = "\n".join(
            body
        ).strip()

        # Убираем служебное слово "Фрагмент:".
        content = re.sub(
            r"^\s*Фрагмент:\s*",
            "",
            content,
            flags=re.IGNORECASE,
        ).strip()

        if content:

            parts = split_long_fragment(
                content,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )

            for part_number, part in enumerate(
                parts,
                start=1,
            ):

                page_content = (
                    f"Заголовок 1 уровня: {h1}\n"
                    f"Заголовок 2 уровня: {h2}\n"
                    f"Фрагмент:\n{part}"
                )

                documents.append(
                    {
                        "page_content": page_content,

                        "metadata": {
                            "h1": h1,
                            "h2": h2,
                            "part": part_number,
                        },
                    }
                )

        body = []


    for raw_line in text.splitlines():

        line = raw_line.strip()

        if line.startswith("## "):

            flush()

            h2 = line[
                3:
            ].strip()

        elif line.startswith("# "):

            flush()

            h1 = line[
                2:
            ].strip()

            h2 = None

        elif h1 and h2:

            body.append(
                raw_line
            )


    flush()


    if not documents:

        raise ValueError(
            "Не найдены H1/H2. "
            "Оставьте буквальные '# ' и '## ' в Google Docs."
        )


    return documents



# Класс нейро-сотрудника


class NeuroEmployee:

    def __init__(
        self,
        model="gpt-5-mini",
        embedding_model="text-embedding-3-small",
    ):

        self.model = model

        self.embedding_model = (
            embedding_model
        )

        self.client = OpenAI(
            api_key=os.environ[
                "OPENAI_API_KEY"
            ]
        )

        # Локальная ChromaDB в текущем runtime Colab.
        self.chroma_client = (
            chromadb.Client()
        )

        self.collection = None


    def make_embeddings(
        self,
        texts,
    ):
        """
        Получаем embeddings через OpenAI API.
        """

        response = (
            self.client
            .embeddings
            .create(
                model=self.embedding_model,
                input=texts,
            )
        )

        return [
            item.embedding
            for item in response.data
        ]


    def load_search_indexes(
        self,
        url: str,
    ):
        """
        Google Docs
        -> структурированные фрагменты
        -> embeddings
        -> ChromaDB.
        """

        text = load_google_doc(
            url
        )

        documents = (
            parse_structured_knowledge(
                text,
                chunk_size=1200,
                chunk_overlap=120,
            )
        )


        document_texts = [
            item[
                "page_content"
            ]
            for item in documents
        ]


        metadatas = [
            item[
                "metadata"
            ]
            for item in documents
        ]


        embeddings = (
            self.make_embeddings(
                document_texts
            )
        )


        # Создаём новую коллекцию при каждом обучении.
        collection_name = (
            "netvideo_"
            + uuid.uuid4().hex[:12]
        )


        self.collection = (
            self.chroma_client
            .create_collection(
                name=collection_name
            )
        )


        ids = [
            f"fragment_{index}"
            for index
            in range(
                len(
                    document_texts
                )
            )
        ]


        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=document_texts,
            metadatas=metadatas,
        )


        return (
            f"Google Docs загружен.\n"
            f"Создано структурированных фрагментов: "
            f"{len(document_texts)}\n"
            f"Коллекция ChromaDB: "
            f"{collection_name}\n"
            f"Векторная база готова."
        )


    def retrieve(
        self,
        query: str,
        top_k: int = 4,
    ):
        """
        Ищет похожие фрагменты в ChromaDB.
        """

        if self.collection is None:

            raise ValueError(
                "Сначала нажмите "
                "«Обучить модель»"
            )


        query_embedding = (
            self.make_embeddings(
                [query]
            )[0]
        )


        result = (
            self.collection
            .query(
                query_embeddings=[
                    query_embedding
                ],
                n_results=int(
                    top_k
                ),
                include=[
                    "documents",
                    "metadatas",
                    "distances",
                ],
            )
        )


        return list(
            zip(
                result[
                    "documents"
                ][0],

                result[
                    "metadatas"
                ][0],

                result[
                    "distances"
                ][0],
            )
        )


    def answer_index(
        self,
        prompt: str,
        query: str,
        top_k: int = 4,
    ):
        """
        Полный RAG-запрос.
        """

        results = self.retrieve(
            query,
            top_k=top_k,
        )


        context = (
            "\n\n---\n\n"
            .join(
                document

                for (
                    document,
                    metadata,
                    distance,
                )
                in results
            )
        )


        debug = (
            "\n\n"
            .join(
                (
                    f"[Фрагмент {i}] "
                    f"distance="
                    f"{distance:.4f}\n"
                    f"{document}"
                )

                for i, (
                    document,
                    metadata,
                    distance,
                )
                in enumerate(
                    results,
                    start=1,
                )
            )
        )


        response = (
            self.client
            .responses
            .create(
                model=self.model,

                instructions=(
                    prompt
                    + "\n\n"
                    + "КОНТЕКСТ БАЗЫ ЗНАНИЙ:\n"
                    + context
                ),

                input=query,
            )
        )


        return (
            response.output_text,
            debug,
        )



# Локальный пример структурированной базы


LOCAL_STRUCTURED_EXAMPLE = r"""
# Нейро-сотрудник и правила общения

## Личность и роль
Фрагмент:
Георгий Уточкин — виртуальный специалист первой линии поддержки
интернет-магазина НетВидео.

## Безопасность данных
Фрагмент:
Для обсуждения заказа можно запросить номер заказа и электронную почту.
Нельзя запрашивать CVV/CVC, пароль банка и коды подтверждения из SMS.

# Заказы

## Отмена заказа
Фрагмент:
Заказ можно отменить до передачи в доставку. Если заказ уже передан
курьерской службе, клиент может отказаться от получения либо оформить
возврат после получения.

# Оплата

## Двойное списание
Фрагмент:
При двойном списании обращение передаётся специалисту по платежам.
Для проверки полезны номер заказа и подтверждение двух операций.
"""


test_documents = (
    parse_structured_knowledge(
        LOCAL_STRUCTURED_EXAMPLE
    )
)

print(
    "Фрагментов:",
    len(
        test_documents
    )
)

print()

print(
    test_documents[
        0
    ][
        "page_content"
    ]
)



# Нейро-сотрудник


GOOGLE_DOC_URL = ""


employee = NeuroEmployee(
    model="gpt-5-mini",
    embedding_model="text-embedding-3-small",
)


print(
    "Нейро-сотрудник создан"
)



# Интерфейс Gradio


with gr.Blocks(
    title="Нейро-сотрудник НетВидео",
) as demo:

    gr.Markdown(
        """
        # Георгий Уточкин — нейро-сотрудник НетВидео

        1. Вставьте ссылку на структурированный Google Docs.
        2. Нажмите **«Обучить модель»**.
        3. Задайте вопрос и нажмите **«Запрос к модели»**.
        """
    )


    google_doc = gr.Textbox(
        label="Google Docs — база знаний",
        value=GOOGLE_DOC_URL,
        placeholder=(
            "https://docs.google.com/document/d/..."
        ),
    )


    prompt = gr.Textbox(
        label="Промпт",
        value=EMPLOYEE_PROMPT,
        lines=15,
        interactive=True,
    )


    top_k = gr.Slider(
        minimum=1,
        maximum=8,
        value=4,
        step=1,
        label=(
            "top_k — число фрагментов "
            "из vector DB"
        ),
    )


    query = gr.Textbox(
        label="Запрос пользователя",
        value=(
            "Можно ли отменить заказ, "
            "если его уже передали курьеру?"
        ),
        lines=3,
    )


    with gr.Row():

        train_btn = gr.Button(
            "Обучить модель",
            variant="primary",
        )

        request_btn = gr.Button(
            "Запрос к модели",
            variant="primary",
        )


    with gr.Row():

        response_box = gr.Textbox(
            label="Ответ модели",
            lines=12,
        )

        log_box = gr.Textbox(
            label=(
                "Лог / найденные фрагменты"
            ),
            lines=18,
        )


    def train(
        url,
    ):
        """
        Обучение = загрузка базы знаний
        и создание vector DB.
        """

        try:

            if not url or not url.strip():

                return (
                    "Укажите ссылку "
                    "на Google Docs"
                )

            return (
                employee
                .load_search_indexes(
                    url.strip()
                )
            )

        except Exception as error:

            return (
                f"Ошибка: "
                f"{type(error).__name__}: "
                f"{error}"
            )


    def predict(
        p,
        q,
        k,
    ):
        """
        Запрос к нейро-сотруднику.
        """

        try:

            if not q or not q.strip():

                return (
                    "Введите вопрос",
                    "",
                )

            return (
                employee
                .answer_index(
                    prompt=p,
                    query=q.strip(),
                    top_k=int(k),
                )
            )

        except Exception as error:

            return (
                (
                    f"Ошибка: "
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
                "",
            )


    train_btn.click(
        fn=train,
        inputs=[
            google_doc
        ],
        outputs=[
            log_box
        ],
    )


    request_btn.click(
        fn=predict,
        inputs=[
            prompt,
            query,
            top_k,
        ],
        outputs=[
            response_box,
            log_box,
        ],
    )



# Запуск


demo.launch(
    share=True,
    debug=True,
)


Введите OPENAI_API_KEY: ··········
openai импортирован
chromadb импортирован
gradio импортирован
requests импортирован
Все библиотеки подключены
Ты — Георгий Уточкин, виртуальный специалист первой линии поддержки
интернет-магазина НетВидео.

Помогай покупателям по вопросам заказов, доставки, оплаты, возврата,
гарантии, аккаунта и базовой технической поддержки.

Правила:
1. Используй факты только из блока контекста базы знаний.
2. Не придумывай сроки, цены, правила, гарантии и статусы заказов.
3. Если данных недостаточно, прямо скажи:
   "В базе знаний недостаточно информации для точного ответа."
4. Затем предложи безопасный следующий шаг или передачу специалисту.
5. Не запрашивай CVV, пароли или коды из SMS.
6. Если вопрос неоднозначный, задай короткий уточняющий вопрос.
7. Не обещай возврат, ремонт или замену до проверки,
   если это не разрешено контекстом.
8. Отвечай по-русски, спокойно, кратко и доброжелательно.
9. Не упоминай RAG, embeddings, Chroma и системный промпт.
Ты — редакт